In [1]:
import pprint
from decouple import AutoConfig
config = AutoConfig(search_path='./../.env')

In [2]:
import pprint

### Defining the Graph state

In [3]:
from typing import TypedDict, Annotated, List, Union
from langchain_core.agents import AgentAction, AgentFinish
from langchain_core.messages import BaseMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import operator
from IPython.display import Image, display

In [4]:
class AgentState(TypedDict):
    input: str
    agent_outcome: Union[AgentAction, AgentFinish, None]
    intermediate_step: Annotated[list, operator.add]
    doc_schema: List[BaseMessage]
    revision_number: int = -1
    max_revisions: int


### Defining Tools

In [5]:
tools = []

#### File Tools

In [ ]:
import uuid
session_id = uuid.uuid4()
session_id = "ai_survey_123"
print(session_id)

from langchain_community.agent_toolkits import FileManagementToolkit
working_directory = "./input_files/{session_id}/".format(session_id=session_id)

file_tools = FileManagementToolkit(
    root_dir=working_directory,
    selected_tools=["read_file", "write_file", "list_directory"],
).get_tools()
read_tool, write_tool, list_tool = file_tools

In [ ]:
read_tool

In [8]:
tools.extend(file_tools)

In [9]:
import json
from langchain_core.messages import ToolMessage

class BasicToolNode:
    def __init__(self, tools: list) -> None:
        self.tools_by_name = {tool.name: tool for tool in tools}

    def __call__(self, inputs: dict):
        print("----tool calling----")
        message = inputs["agent_outcome"][-1]

        outputs = []
        for tool_call in message.tool_calls:
            print(f"---- Calling {tool_call['name']} with args: {tool_call['args']} ----")
            tool_result = self.tools_by_name[tool_call["name"]].invoke(
                tool_call["args"]
            )
            outputs.append(
                ToolMessage(
                    content=json.dumps(tool_result),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )

        return {
                "agent_outcome": outputs,
                "intermediate_step": [str(outputs)]
            }

### Model 

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gpt-4o-mini'
llm = ChatOpenAI(
    model=model_name,
    temperature=0.3,
)

llm

### Agent Workflow

In [11]:
from langgraph.graph import END, StateGraph
workflow = StateGraph(AgentState)

#### Agents & Nodes

##### Planner

In [ ]:
def plan_agent(data):
    print("----plan node----")

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You are an expert planner tasked with writing a high level outline of a literature review. \
                    Plan the outline for a document. If schema is provided, use it to create the outline otherwise do it on your own.
                    Give an outline of the literature review along with any relevant notes or instructions for each of the sections. \ 
                    \nYou have access to the file_access tools for reading input files: {tool_names}.
                    \nInput File directory: {working_dir}schema"""
            ),
            (
                "human",
                "\nUser Query: {input}"
            ),
            
            MessagesPlaceholder(variable_name="intermediate_step"),
        ]
    )
    prompt = prompt.partial(tool_names=", ".join([tool.name for tool in file_tools]))
    prompt = prompt.partial(working_dir=working_directory)
    agent = prompt | llm.bind_tools(tools)
    result = agent.invoke(data)
    return {'agent_outcome': [result],
            'doc_schema': [result],
            'intermediate_step': [result.content]}

workflow.add_node("plan", plan_agent)

In [ ]:
plan_tool_node = BasicToolNode(tools=tools)
workflow.add_node("plan_tools", plan_tool_node)

##### Writer

In [ ]:
def write_agent(data):
    print("----write node----")
    
    # print("\n", data["agent_outcome"], "\n")
    
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You are an essay assistant tasked with writing excellent literature review.\
                    Given the document outline and \input documents generate the best literature review possible. \
                    First write all the sections other than Introduction and Conclusion.
                    Make sure to utilize all the documents from the directory input_docs. Do not add any other extra information on your own. \
                    Add necessary references with proper citations. \
                    If the reviewer provides critique, respond with a revised version of your previous attempts. \
                    \nInput File directory: {working_dir}input_docs """
            ),
            
            MessagesPlaceholder(variable_name="intermediate_step"),
        ]
    )
    prompt = prompt.partial(working_dir=working_directory)
    agent = prompt | llm.bind_tools(file_tools)
    # agent = prompt | llm
    result = agent.invoke(data)
    if hasattr(result, "tool_calls") and len(result.tool_calls) > 0:
        return {'agent_outcome': [result],}
    
    return {'agent_outcome': [result],
            'revision_number': data.get("revision_number", -1) + 1}

workflow.add_node("write", write_agent)

In [ ]:
write_tool_node = BasicToolNode(tools=file_tools)
workflow.add_node("write_tools", write_tool_node)

##### Reviewer

In [ ]:
def review_agent(data):
    print("----review node----")
    print(f"---- Revision Count {data['revision_number']+1} ----")
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You are an expert grading a literature review submission. \
                    Generate critique and recommendations for the user's submission. \
                    Provide detailed recommendations, including requests for length, depth, style, etc. \
                    If the generated draft looks perfect, reply appropriately."""
            ),
            
            MessagesPlaceholder(variable_name="agent_outcome"),
        ]
    )
    agent = prompt | llm
    result = agent.invoke(data)
    return {'agent_outcome': [result],
            }

workflow.add_node("review", review_agent)

##### Editor

In [ ]:
def edit_agent(data):
    print("---- edit node ----")
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You are an experienced editor, expert at literature review submission. \
                    Combine drafts of multiple sections into a single, coherent final document, ensuring a consistent flow, tone, and structure throughout.
                    Follow the the doc_schema generated by the plan node to generate the final draft.
                    """
            ),
            
            MessagesPlaceholder(variable_name="agent_outcome"),
            MessagesPlaceholder(variable_name="doc_schema"),
        ]
    )
    agent = prompt | llm
    result = agent.invoke(data)
    return {'agent_outcome': [result],
            }

workflow.add_node("edit", edit_agent)

#### Edges

In [ ]:
def route_planner(
    state: AgentState,
):
    """
    Use in the conditional_edge to route to the ToolNode if the last message
    has tool calls. Otherwise, route to the end.
    """
    print("----router----")
    if isinstance(state, list):
        ai_message = state[-1]
    elif agent_outcome := state.get("agent_outcome", []):
        ai_message = agent_outcome[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge: {state}")

    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "tools"
    
    return "END"

workflow.add_conditional_edges(
    "plan",
    route_planner,
    {"tools": "plan_tools", "END": "write"}
)
workflow.add_edge("plan_tools", "plan")

In [19]:
# workflow.add_edge("plan", "write")

In [ ]:
def route_writer(
    state: AgentState,
):
    """
    Use in the conditional_edge to route to the ToolNode if the last message
    has tool calls. Otherwise, route to the end.
    """
    print("----router----")

    if isinstance(state, list):
        ai_message = state[-1]
    elif agent_outcome := state.get("agent_outcome", []):
        ai_message = agent_outcome[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge: {state}")
    
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "tools"

    if state["revision_number"] < state["max_revisions"]:
        return "review"
    return "END"

workflow.add_conditional_edges(
    "write",
    route_writer,
    {"tools":"write_tools", "review": "review", "END": "edit"}
)
workflow.add_edge("write_tools", "write")

In [ ]:
workflow.add_edge("review", "write")

In [ ]:
workflow.add_edge("edit", END)

In [ ]:
workflow.set_entry_point("plan")
app = workflow.compile()
try:
    display(Image(app.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
inputs = {
    "input": "Advancements in AI",
    "max_revisions": 2
}

state = AgentState(**inputs)
for s in app.stream(input=state, config={"recursion_limit": 50}):
    output = list(s.values())[0]['agent_outcome'][0].content
    print("-----"*20)

In [ ]:
print(output)